## Output Parsers in LangChain

Output Parsers are specialized tools that **transform raw LLM responses into structured formats**.  
They ensure that the model’s output is predictable, validated, and ready for downstream use.

---

### StringOutputParser
- **Purpose:** Extracts plain text from the LLM response.  
- **Best For:** Summaries, explanations, or free-form answers.  
- **Example Use Case:** When you just need the model’s reply as a string without any structure.



### JsonOutputParser
- **Purpose:** Parses the LLM output into a JSON object (dictionary).  
- **Best For:** When you need structured data but don’t want strict validation.  
- **Example Use Case:** Extracting product details, user profiles, or configuration data in JSON format.



### PydanticOutputParser
- **Purpose:** Parses output into a **Pydantic model** with validation.  
- **Best For:** Applications requiring **type safety** and **automatic coercion** (e.g., `"128"` → `128`).  
- **Example Use Case:** Validating API responses, enforcing schema in production systems.



### MarkdownListOutputParser
- **Purpose:** Extracts bullet lists (`- item`) from Markdown into a Python list.  
- **Best For:** Enumerations, options, or step-by-step instructions.  
- **Example Use Case:** Generating a list of tasks, categories, or ideas in Markdown format.

---

##  Takeaway
- Use **StringOutputParser** for free text.  
- Use **JsonOutputParser** for flexible structured data.  
- Use **PydanticOutputParser** when strict validation and type safety are required.  
- Use **MarkdownListOutputParser** when you want clean lists extracted from Markdown.  

## StringOutputParser

- **Purpose:**  
  The `StringOutputParser` is the simplest output parser in LangChain.  
  It takes the raw response from the LLM and **extracts plain text only**.

- **Why Use It?**  
  - When you don’t need structured data, just the text itself.  
  - Ideal for summaries, explanations, or answers in natural language.  
  - Avoids schema validation errors since it doesn’t enforce any structure.

- **Best Situation to Use:**  
  - Quick prototypes.  
  - Educational demos where you want to show the raw text output.  
  - When the downstream application only needs a string (e.g., chatbot replies).

In [ ]:
# Simplest StringOutputParser Example with Groq
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

# Load environment variables (make sure GROQ_API_KEY is in your .env file)
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant", 
                api_key=groq_key)

# Define a simple prompt
prompt = PromptTemplate.from_template("Write a two-sentence mystery story.")

# Attach StringOutputParser
chain = prompt | llm | StrOutputParser()

# Invoke the chain
response = chain.invoke({})

# Print plain text output
print("LLM Response:", response)

LLM Response: Detective Jameson stood at the edge of the abandoned mansion, where a single rose had been left on the doorstep with a note that read: "The truth is within." As he stepped inside, he heard the faint whisper of his own name echoing through the empty halls, but there was no one in sight.


### Flow of `prompt | llm | StrOutputParser()`

Before diving into why it’s called a "chain," let’s visualize the **flow**:

1. **PromptTemplate (`prompt`)**  
   - Takes your input (e.g., `"Write a two-sentence mystery story."`)  
   - Formats it into a structured prompt for the LLM.

2. **LLM (`llm`)**  
   - Receives the formatted prompt.  
   - Generates a raw response (could be text, JSON, or other formats depending on setup).

3. **StrOutputParser()**  
   - Extracts **plain text only** from the LLM response.  
   - Removes metadata, ensures you get a clean string.

---

## Why It’s Called a "Chain"

- In **LangChain’s older API**, everything was called a **Chain** (e.g., `LLMChain`, `SequentialChain`).  
- In the **newer design**, these are unified under **Runnables**:
  - Each component (Prompt, LLM, Parser) is a **Runnable**.
  - When you connect them with the pipe operator `|`, you create a **RunnableSequence**.

---

### Why People Still Say "Chain"
- **Historical naming:** Developers are used to saying "chain" because that was the original abstraction.  
- **RunnableSequence ≈ Chain:** Functionally, it behaves like a chain of steps, so the term stuck.  
- **Backward compatibility:** Many tutorials and docs still use "chain" for simplicity, even though the underlying class is now `RunnableSequence`.

---

### Takeaway
- `prompt | llm | StrOutputParser()` is **technically a Runnable pipeline**.  
- It’s often called a **chain** informally because it represents a sequence of steps.  
- Think of it like this:  
  - **Old world:** Chains were recipes.  
  - **New world:** Runnables are Lego blocks.  
  - When you snap Lego blocks together, you still end up with a "chain" of steps.

  ---
  ---


## 🔹 JsonOutputParser 

- **Purpose:**  
  The `JsonOutputParser` is used to **parse LLM outputs into JSON objects** (Python dictionaries).  
  It ensures the response is structured and machine-readable, without enforcing strict type validation.

- **Why Use It?**  
  - Converts free-form text into structured JSON.  
  - Flexible: avoids strict type errors (e.g., `"82"` vs `82`).  
  - Ideal for scenarios where you want structured data but can handle type conversions yourself.

- **Best Situations to Use:**  
  - APIs or applications that exchange JSON data.  
  - Cross-language projects where JSON is the common format.  
  - Sports, recipes, or any domain where structured fields are needed.

- **Note on Cricket Example:**  
  - You can ask the LLM to output match details (like runs, venue, teams) in JSON format.  
  - Even if runs are returned as `"82"` (string), you can later convert them to integers in Python.


In [4]:
# Cricket Example with JsonOutputParser using Groq
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Load environment variables (make sure GROQ_API_KEY is in your .env file)
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Define a prompt asking for cricket match summary in JSON format
prompt = PromptTemplate.from_template(
    "Give me a JSON object with keys: team_won, team_lost, venue, top_batter, top_batter_runs, key_moment "
    "for a fictional IPL 2026 final between Mumbai Indians and RCB."
)

# Attach JsonOutputParser
chain = prompt | llm | JsonOutputParser()

# Invoke the chain
response = chain.invoke({})

# Print parsed JSON output
print("Parsed Cricket JSON:", response)
print("Winner:", response["team_won"])
print("Venue :", response["venue"])
print("Top Batter:", response["top_batter"], "—", response["top_batter_runs"], "runs")
print("Key Moment:", response["key_moment"])

Parsed Cricket JSON: {'team_won': 'Mumbai Indians', 'team_lost': 'RCB', 'venue': 'Dubai International Cricket Stadium, Dubai', 'top_batter': 'Suryakumar Yadav', 'top_batter_runs': 84, 'key_moment': "Suryakumar Yadav's 84-run knock off 45 balls and Trent Boult's 3-wicket haul helped Mumbai Indians secure a 15-run victory over RCB, clinching their 6th IPL title."}
Winner: Mumbai Indians
Venue : Dubai International Cricket Stadium, Dubai
Top Batter: Suryakumar Yadav — 84 runs
Key Moment: Suryakumar Yadav's 84-run knock off 45 balls and Trent Boult's 3-wicket haul helped Mumbai Indians secure a 15-run victory over RCB, clinching their 6th IPL title.


## Handling Missing Values in JsonOutputParser

- **JsonOutputParser** is flexible and parses whatever JSON the LLM provides.  
- However, the LLM may sometimes **skip fields** or return `None`/`null`.  
- Example: Instead of `"top_batter_runs": 84`, the model might output `"top_batter_runs": null`.  
- This means your code must handle missing values gracefully to avoid crashes.

---

### Why This Happens
- LLMs are probabilistic: they don’t always fill every key.  
- JsonOutputParser doesn’t enforce strict validation, it just parses the JSON.  
- If a field is missing, you’ll get `None` or the key may not exist at all.

---

### How to Handle It
- Use safe access methods in Python:
  - `response.get("top_batter_runs", "N/A")` → avoids `KeyError`.  
  - Convert values safely: `int(response.get("top_batter_runs", 0))`.  
- Add fallback defaults (e.g., `"N/A"`, `0`, `"Unknown"`).  
- This ensures your app doesn’t crash when the LLM skips a value.

---

## When to Use JsonOutputParser
- Use it when:
  - You want **flexible structured data**.  
  - You can tolerate missing values and handle them in code.  
  - You’re building prototypes, demos, or apps where strict validation isn’t critical.

---

## Alternatives for Strict Validation
- If you need **guaranteed fields** and strict type safety:
  - **PydanticOutputParser** → enforces schema, validates types, auto‑coerces values.  
  - **with_structured_output(PydanticModel)** → ensures the LLM output matches the model exactly.  
- These options are better for **production systems** where missing or invalid values could break the workflow.

---

### Takeaway
- **JsonOutputParser** = flexible, but you must handle `None` values yourself.  
- **PydanticOutputParser / Structured Output** = strict, catches errors early, prevents crashes.  
- Choose based on your project needs:  
  - **Prototyping → JsonOutputParser**  
  - **Production → PydanticOutputParser**


## MarkdownListOutputParser

- **Purpose:**  
  The `MarkdownListOutputParser` is a simple parser that extracts **bullet lists** (`- item` or `* item`) from Markdown text and converts them into a **Python list of strings**.  

- **Why Use It?**  
  - When you want the LLM to output a clean list of items in Markdown.  
  - Automatically converts bullet points into a usable Python list.  
  - Lightweight and avoids schema overhead — no validation, just list extraction.

- **Best Situations to Use:**  
  - Brainstorming ideas (e.g., "List 5 startup ideas").  
  - Study notes where students need a list of key points.  
  - Sports highlights, recipe ingredients, or cricket match moments formatted as bullet points.  

- **Key Difference:**  
  - **MarkdownListOutputParser** → extracts only bullet lists into a Python list.  
  - **JsonOutputParser** → parses full JSON objects.  
  - **PydanticOutputParser** → enforces strict schema validation.  

---

### Example Flow
1. **Prompt** → Ask the LLM to reply only as a Markdown bullet list.  
2. **LLM Response** → Produces bullet points in Markdown.  
3. **MarkdownListOutputParser** → Converts them into a Python list.  

---


In [8]:
# Example: MarkdownListOutputParser with Groq
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.output_parsers import MarkdownListOutputParser

# Load environment variables (make sure GROQ_API_KEY is in your .env file)
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Attach MarkdownListOutputParser
parser = MarkdownListOutputParser()
chain = llm | parser

# Prompt asking for cricket match highlights in Markdown bullet list
prompt = """
List 3 key highlights of a fictional IPL 2026 final between Mumbai Indians and RCB.
Reply only as a Markdown bullet list.
"""

# Invoke the chain
response = chain.invoke(prompt)

# Print parsed Python list
print("Parsed List Output:", response)

# Print in Markdown style (one per line)
print("\n### Cricket Highlights\n")
for item in response:
    print(f"- {item}")


Parsed List Output: ['Rohit Sharma leads Mumbai Indians to a thrilling 3-run victory over RCB in the IPL 2026 final.', "The match was decided in the final over, with Mumbai Indians' pacer, Jasprit Bumrah, securing the win by dismissing RCB's Faf du Plessis.", 'Virat Kohli scored a valiant 92 off 45 deliveries for RCB, but it was not enough to help his team overcome a strong 180-run total from Mumbai Indians.']

### Cricket Highlights

- Rohit Sharma leads Mumbai Indians to a thrilling 3-run victory over RCB in the IPL 2026 final.
- The match was decided in the final over, with Mumbai Indians' pacer, Jasprit Bumrah, securing the win by dismissing RCB's Faf du Plessis.
- Virat Kohli scored a valiant 92 off 45 deliveries for RCB, but it was not enough to help his team overcome a strong 180-run total from Mumbai Indians.


## MarkdownOutputParser

- **Purpose:**  
  The `MarkdownOutputParser` is designed to parse **Markdown-formatted text** into structured Python objects.  
  Unlike `MarkdownListOutputParser` (which only extracts bullet lists), this parser can handle **headings, lists, and text blocks**.

- **Why Use It?**  
  - When you want the LLM to output in Markdown and then convert that into structured data.  
  - Useful for scenarios where sections, headings, and lists need to be preserved.  
  - Ideal for educational notes, travel itineraries, recipes, or reports where Markdown formatting is natural.

- **Best Situations to Use:**  
  - Study notes with headings and bullet points.  
  - Travel plans broken into days with activities.  
  - Reports or documentation where Markdown is the preferred format.  

- **Key Difference:**  
  - **MarkdownListOutputParser** → extracts only bullet lists into a Python list.  
  - **MarkdownOutputParser** → parses full Markdown documents (headings + lists + text) into structured Python objects.

In [9]:
# Simplest MarkdownListOutputParser Example with Groq
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.output_parsers import MarkdownListOutputParser

# Load environment variables (make sure GROQ_API_KEY is in your .env file)
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Attach MarkdownListOutputParser
parser = MarkdownListOutputParser()
chain = llm | parser

# Prompt asking for cricket match highlights in Markdown bullet list
prompt = """
Give me three bullet points in Markdown about a fictional IPL 2026 final between Mumbai Indians and RCB.
"""

# Invoke the chain
response = chain.invoke(prompt)

# Print parsed Python list
print(response)

["**Match Result**: Mumbai Indians emerged victorious, winning the match by 22 runs. The target of 190 runs set by Mumbai Indians was successfully chased by RCB but for a brief moment, Mumbai Indians seemed almost unbeatable after Rohit Sharma's aggressive start to the game.", '**Player of the Match**: Rohit Sharma was awarded the player of the match for his exceptional performance. He scored 73 runs in 45 balls and also took 2 crucial wickets to seal the fate of the match for Mumbai Indians.', "**MVP of IPL 2026**: Although the season has just started, Rohit Sharma's incredible display in the IPL 2026 final has made him a strong contender for the Most Valuable Player award of the season."]


In [10]:
# Print each highlight on a new line
print("Parsed Cricket Highlights:")
for item in response:
    print("-", item)

Parsed Cricket Highlights:
- **Match Result**: Mumbai Indians emerged victorious, winning the match by 22 runs. The target of 190 runs set by Mumbai Indians was successfully chased by RCB but for a brief moment, Mumbai Indians seemed almost unbeatable after Rohit Sharma's aggressive start to the game.
- **Player of the Match**: Rohit Sharma was awarded the player of the match for his exceptional performance. He scored 73 runs in 45 balls and also took 2 crucial wickets to seal the fate of the match for Mumbai Indians.
- **MVP of IPL 2026**: Although the season has just started, Rohit Sharma's incredible display in the IPL 2026 final has made him a strong contender for the Most Valuable Player award of the season.
